# Paligemma

[paligemma](pics/paligemma.png)

---

## 一、Configuration

In [ ]:
import torch
import torch.nn as nn


class SiglipVisionConfig:
    def __init__(
        self,
        hidden_size=768,
        intermediate_size=3072,
        num_hidden_layers=12,
        num_q_heads=12,
        num_channels=3,  # RGB三通道
        image_size=224,  # resize后的图像尺寸
        patch_size=16,   # 切割得到一个patch的边长
        layer_norm_eps=1e-16,
        attention_dropout=0.0,
        num_image_tokens: int = None,
        **kwargs,
    ):
        super().__init__()
        self.hidden_size = hidden_size
        self.intermediate_size = intermediate_size
        self.num_hidden_layers = num_hidden_layers
        self.num_q_heads = num_q_heads
        self.num_channels = num_channels
        self.image_size = image_size
        self.patch_size = patch_size
        self.layer_norm_eps = layer_norm_eps
        self.attention_dropout = attention_dropout
        self.num_image_tokens = num_image_tokens
        

class GemmaConfig:
    def __init__(
        self,
        vocab_size,
        hidden_size,
        intermediate_size,
        num_hidden_layers,
        num_q_heads,
        num_kv_heads,  # Grouped-Query Attention
        head_dim=256,
        max_position_embeddings=8192,
        rms_norm_eps=1e-6,
        rope_theta=10000.0,
        attention_bias=False,
        attention_dropout=0.0,
        pad_token_id=None,
        **kwargs,
    ):
        super().__init__()
        self.vocab_size = vocab_size
        self.max_position_embeddings = max_position_embeddings
        self.hidden_size = hidden_size
        self.intermediate_size = intermediate_size
        self.num_hidden_layers = num_hidden_layers
        self.num_q_heads = num_q_heads
        self.num_kv_heads = num_kv_heads
        self.head_dim = head_dim
        self.rms_norm_eps = rms_norm_eps
        self.rope_theta = rope_theta
        self.attention_bias = attention_bias
        self.attention_dropout = attention_dropout
        self.pad_token_id = pad_token_id
        
        
class PaliGemmaConfig:
    def __init__(
        self,
        vision_config=None,
        text_config=None,
        ignore_index=-100,
        image_token_index=256000,
        vocab_size=257152,
        projection_dim=2048,  # projection的输出维度
        hidden_size=2048,     # 语言模型的嵌入维度
        pad_token_id=None,
        **kwargs,
    ):
        super().__init__()
        self.ignore_index = ignore_index
        self.image_token_index = image_token_index
        self.vocab_size = vocab_size
        self.projection_dim = projection_dim
        self.hidden_size = hidden_size
        self.vision_config = vision_config
        self.is_encoder_decoder = False
        self.pad_token_id = pad_token_id

        self.vision_config = SiglipVisionConfig(**vision_config)
        self.text_config = GemmaConfig(**text_config, pad_token_id=pad_token_id)
        self.vocab_size = self.text_config.vocab_size

        self.text_config.num_image_tokens = (self.vision_config.image_size // self.vision_config.patch_size)**2
        self.vision_config.projection_dim = projection_dim

---

## 二、SigLIP Embedding

In [ ]:
class SiglipVisionEmbeddings(nn.Module):
    def __init__(self, config: SiglipVisionConfig):
        super().__init__()
        self.config = config
        self.embed_dim = config.hidden_size
        self.image_size = config.image_size
        self.patch_size = config.patch_size
        
        """
        kernel_size = stride = patch_size, 卷积步之间无重叠
        保证输出的特征图长宽正好均为 image_size//stride=num_patches_per_dim
        """
        self.patch_embedding = nn.Conv2d(
            in_channels=config.num_channels,
            out_channels=self.embed_dim,
            kernel_size=self.patch_size,
            stride=self.patch_size,
            padding="valid",  # no padding
        )
        
        self.num_patches = (self.image_size // self.patch_size) ** 2  # 将正方形切割为patch的集合
        self.num_positions = self.num_patches
        
        self.position_embedding = nn.Embedding(self.num_positions, self.embed_dim)  # learnable, 训练时更新
        self.register_buffer(
            "position_idx",
            torch.arange(self.num_positions).unsqueeze(0),
            persistent=False,
        )
        
    def forward(self, pixel_values: torch.Tensor) -> torch.Tensor:
        """
        通过卷积操作计算输入pixels的嵌入向量, 获取patch_embeds:
        (batch, channels, height, width) -> (batch, embed_dim, num_patches_H, num_patches_W)
        再将patch_embeds展平并将num_patches维往前提:
        (batch, embed_dim, num_patches_H, num_patches_W) -> (batch, embed_dim, num_patches) -> (batch, num_patches, embed_dim)
        """
        patch_embeds = self.patch_embedding(pixel_values)
        embeddings = patch_embeds.flatten(2).transpose(1, 2)
        
        """
        position_embedding: (1, num_patches) -> (1, num_patches, embed_dim)
        将线性位置编码叠加到patch_embeds上, 最终得到: (batch, num_patches, embed_dim)
        """
        embeddings = embeddings + self.position_embedding(self.position_idx)
        return embeddings

---

## 三、SigLIP VisionTransformer

In [ ]:
import torch.nn.functional as F

class SiglipMLP(nn.Module):
    def __init__(self, config: SiglipVisionConfig):
        super().__init__()
        self.fc1 = nn.Linear(config.hidden_size, config.intermediate_size)
        self.fc2 = nn.Linear(config.intermediate_size, config.hidden_size)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        (batch, num_patches, embed_dim) -> (batch, num_patches, intermediate_size) -> (batch, num_patches, embed_dim)
        """
        x = self.fc1(x)
        x = F.gelu(x, approximate="tanh")
        x = self.fc2(x)
        return x


class SiglipAttention(nn.Module):
    def __init__(self, config: SiglipVisionConfig):
        super().__init__()
        self.config = config
        self.embed_dim = config.hidden_size
        self.num_heads = config.num_attention_heads
        self.head_dim = self.embed_dim // self.num_heads
        self.scale = self.head_dim ** -0.5  # 1 / sqrt(head_dim)
        self.dropout = config.attention_dropout
        
        self.q_proj = nn.Linear(self.embed_dim, self.embed_dim)
        self.k_proj = nn.Linear(self.embed_dim, self.embed_dim)
        self.v_proj = nn.Linear(self.embed_dim, self.embed_dim)
        self.out_proj = nn.Linear(self.embed_dim, self.embed_dim)
    
    """
    区别于LLM中的causal attention, SiglipAttention建立了所有patches之间的联系
    """
    def forward(self, x: torch.Tensor):
        batch_size, num_patches, embed_dim = x.size()
        """
        将同一个patched input分别投影至query, key, value, 形状不改变
        (batch, num_patches, embed_dim) -> (batch, num_patches, head_dim)
        """
        query = self.q_proj(x)
        key   = self.k_proj(x)
        value = self.v_proj(x)
        
        """
        将Q, K, V分别拆分成多个head, 再前置多头位置使每个head都可以看到所有的patches(的部分embed)
        (batch, num_patches, embed_dim) -> (batch, num_heads, num_patches, head_dim)
        """
        query = query.view(batch_size, self.num_heads, num_patches, self.head_dim)
        key   = key.view(batch_size, self.num_heads, num_patches, self.head_dim)
        value = value.view(batch_size, self.num_heads, num_patches, self.head_dim)
        
        """
        将多头Q乘以转置的K, 并作用softmax计算注意力分数:
        (batch, num_heads, num_patches, head_dim) @ (batch, num_heads, head_dim, num_patches) -> (batch, num_heads, num_patches, num_patches)
        再乘以多头V, 得到多头重建patches:
        (batch, num_heads, num_patches, num_patches) @ (batch, num_heads, num_patches, head_dim) -> (batch, num_heads, num_patches, head_dim)
        """
        attn_score  = torch.matmul(query, key.transpose(-2, -1)) * self.scale
        attn_score  = F.softmax(attn_score, dim=-1, dtype=torch.float32)
        attn_score  = F.dropout(attn_score, p=self.dropout, training=self.training)
        attn_output = torch.matmul(attn_score, value)
        
        """
        将patches拼接重建成原始形状:
        (batch, num_heads, num_patches, head_dim) -> (batch, num_patches, num_heads * head_dim)
        再通过out_proj映射到输出空间, 形状不变:
        (batch, num_patches, num_heads * head_dim) -> (batch, num_patches, embed_dim)
        """
        attn_output = attn_output.transpose(1, 2).contiguous()
        attn_output = attn_output.reshape(batch_size, num_patches, self.embed_dim)
        attn_output = self.out_proj(attn_output)
        
        return attn_output
    
    
class SiglipEncoderLayer(nn.Module):
    def __init__(self, config: SiglipVisionConfig):
        super().__init__()
        self.embed_dim = config.hidden_size
        self.self_attn = SiglipAttention(config)
        self.layer_norm1 = nn.LayerNorm(config.hidden_size, eps=config.layer_norm_eps)
        self.mlp = SiglipMLP(config)
        self.layer_norm2 = nn.LayerNorm(config.hidden_size, eps=config.layer_norm_eps)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Block1: (batch, num_patches, embed_dim) -> (batch, num_patches, embed_dim), 形状始终不变
        """
        residual = x
        x = self.layer_norm1(x)
        x = self.self_attn(x)
        x = x + residual
        
        """
        Block2: (batch, num_patches, embed_dim) -> (batch, num_patches, embed_dim), 同上
        """
        residual = x
        x = self.layer_norm2(x)
        x = self.mlp(x)  # 扩充模型参数 + 引入非线性性, 支持建模更加复杂的关系
        x = x + residual
        
        return x


class SiglipEncoder(nn.Module):
    def __init__(self, config: SiglipVisionConfig):
        super().__init__()
        self.config = config
        self.layers = nn.ModuleList(
            [SiglipEncoderLayer(config) for _ in range(config.num_hidden_layers)]
        )
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        经过多个EncoderLayer, 形状始终不变: (batch, num_patches, embed_dim)
        """
        for encoder_layer in self.layers:
            x = encoder_layer(x)
        return x
        
    
class SiglipVisionTransformer(nn.Module):
    def __init__(self, config: SiglipVisionConfig):
        super().__init__()
        self.config = config
        embed_dim = config.hidden_size
        self.embeddings = SiglipVisionEmbeddings(config)
        self.encoder = SiglipEncoder(config)
        self.post_layernorm = nn.LayerNorm(embed_dim, eps=config.layer_norm_eps)
        
    def forward(self, pixel_values: torch.Tensor) -> torch.Tensor:
        """
        将输入图像转化为patch_embedding
        (batch, channels, height, width) -> (batch, num_patches, embed_dim)
        """
        hidden_states = self.embeddings(pixel_values)
        
        last_hidden_states = self.encoder(hidden_states)
        last_hidden_states = self.post_layernorm(last_hidden_states)
        
        return last_hidden_states
    
    
class SiglipVisionModel(nn.Module):
    def __init__(self, config: SiglipVisionConfig):
        super().__init__()
        self.config = config
        self.vit = SiglipVisionTransformer(config)
    
    def forward(self, pixel_values: torch.Tensor) -> torch.Tensor:
        """
        输入图像像素值, 输出图像嵌入特征
        (batch, channels, height, width) -> (batch, num_patches, embed_dim)
        """
        vit_output = self.vit(pixel_values=pixel_values)
        return vit_output

---

## 四、Processing

In [ ]:
from PIL import Image
import numpy as np
from typing import List

IMAGENET_STANDARD_MEAN = [0.5, 0.5, 0.5]
IMAGENET_STANDARD_STD = [0.5, 0.5, 0.5]

class PaliGemmaProcessor:    
    def __init__(self, tokenizer, num_image_tokens: int, image_size: int):
        super().__init__()
        self.tokenizer = tokenizer
        self.num_image_tokens = num_image_tokens
        self.image_size = image_size
        self.IMAGE_TOKEN = 0
        
        self.image_token_id = tokenizer.convert_tokens_to_ids(self.IMAGE_TOKEN)
        self.tokenizer.add_bos_token = False
        self.tokenizer.add_eos_token = False

    def process_images(
        self,
        images: Image.Image,
        size: tuple,
        resample: Image.Resampling = None,
        rescale_factor: float = None,
        image_mean: List[float] = None,
        image_std: List[float] = None,
    ):
        mean, std = np.array(image_mean, dtype=np.float32), np.array(image_std, dtype=np.float32)
        images = [image.resize(size=(size[0], size[1]), resample=resample) for image in images]
        images = [np.array(image).astype(np.float32) for image in images]
        images = [image * rescale_factor for image in images]
        images = [(image - mean) / std for image in images]
        images = [image.transpose(2, 0, 1) for image in images]  # (height, width, channels) -> (channels, height, width)
        
        pixel_values = np.stack(images, axis=0)  # (batch, channels, height, width)
        return torch.tensor(pixel_values, dtype=torch.float32)
    
    def add_image_tokens_to_prompt(
        prefix: str,
        bos_token: str,
        image_token: str,
        num_image_tokens: int,
    ):
        """
        构建预设提示, 包含image placeholder
        PaliGemma的实现形式为: <image>...<image><bos>[prompt]\n
        """
        return f"{image_token * num_image_tokens}{bos_token}{prefix}\n"
    
    def __call__(
        self,
        texts: List[str],
        images: List[Image.Image],
        padding: str = "longest",
        truncation: bool = True,
    ):
        assert len(texts) == 1 and len(images) == 1, "Only one text and one image are supported."
        
        """
        将PIL图像转化为pixel tensor, 注意前后两者的*通道位置*不一样
        """
        pixel_values = self.process_images(
            images=images,
            size=(self.image_size, self.image_size),
            resample=Image.Resampling.BICUBIC,
            rescale_factor=1 / 255.0,
            image_mean=IMAGENET_STANDARD_MEAN,
            image_std=IMAGENET_STANDARD_STD,
        )
        
        """
        构建预设提示, 将image token插入text token中
        """
        input_strings = [
            self.add_image_tokens_to_prompt(
                prefix=prompt,
                bos_token=self.tokenizer.bos_token,
                image_token=self.IMAGE_TOKEN,
                num_image_tokens=self.num_image_tokens,
            )
            for prompt in texts
        ]
        
        """
        将文本和图像的输入字符转化为token id, 并以tensor形式返回 input_ids 和 attention_mask
        """
        inputs = self.tokenizer(
            input_strings,
            padding=padding,
            truncation=truncation,
            return_tensors="pt",
        )
        
        return_data = {"pixel_values": pixel_values, **inputs}
        return return_data

---

## 五、Gemma LLM

In [ ]:
class KVCache:
    pass